In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys
import datetime
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

sys.path.append('/home/wolfgang/repos/sleep_general')
from mgh_sleeplab import load_mgh_signal, annotations_preprocess, vectorize_respiratory_events, vectorize_sleep_stages

In [2]:
table_grass = pd.read_csv('/media/mad3/Datasets_ConvertedData/sleeplab/grass_studies_list.csv')
table_natus = pd.read_csv('/media/mad3/Datasets_ConvertedData/sleeplab/natus_studies_list.csv')
assert np.all(table_grass.columns == table_natus.columns)
sleeplab_table = pd.concat([table_grass, table_natus], axis=0)
sleeplab_table = sleeplab_table[pd.notna(sleeplab_table.MRN)]
sleeplab_table.reset_index(inplace=True, drop=True)
print(sleeplab_table.shape)

(25231, 10)


In [3]:
print('TMP')
part = 0
sleeplab_table = sleeplab_table.iloc[8700*part : 8700*(part+1)].copy()
print(sleeplab_table.shape)

TMP
(8700, 10)


In [4]:
sleeplab_table['path_signal'] = np.nan
sleeplab_table['path_annotations'] = np.nan

for jloc, row in sleeplab_table.iterrows():
    
    try:
        path = row.Path
        path = path.replace('M:', '/media/mad3')
        path = path.replace('\\', '/')

        path_signalfile = np.nan
        signalfile = [x for x in os.listdir(path) if 'Signal' in x]
        if len(signalfile) > 0:
            signalfile = signalfile[0]
            path_signalfile = os.path.join(path, signalfile)

        path_annotations = np.nan
        if 'annotations.csv' in os.listdir(path):
            path_annotations = os.path.join(path, 'annotations.csv')

        sleeplab_table.loc[jloc, 'path_signal'] = path_signalfile
        sleeplab_table.loc[jloc, 'path_annotations'] = path_annotations

    except Exception as e:
        print(e)
        continue
        
sleeplab_table.to_csv(f'sleeplab_table_{part}.csv')

In [5]:
print('TMP')
sleeplab_table = sleeplab_table.iloc[:8000].copy()
print(sleeplab_table.shape)

TMP
(8000, 12)


In [6]:
from sleep_analysis_functions import compute_spo2_clean, compute_hypoxia_burden, desaturation_detection, hypoxaemic_burden_minutes

def compute_hypoxia_statistics(data, apnea, sleep_stage, fs):
    
    data['apnea'] = apnea

    dt_start = pd.Timestamp('2000-01-01 00:00:00')
    dt_end = dt_start + datetime.timedelta(seconds=(data.shape[0]-1) / fs)
    pseudo_dt_index = pd.date_range(start=dt_start, end=dt_end, periods=data.shape[0])
    data.index = pseudo_dt_index

    data = compute_spo2_clean(data, fs=fs)
    data['spo2'] = data['spo2_clean']

    data['apnea_binary'] = np.isin(data['apnea'],[1,2,3,4]).astype(int)
    data['apnea_end'] = np.isin(data['apnea_binary'].diff(), [-1])
    
    sleep_stage = sleep_stage[np.logical_not(pd.isna(sleep_stage))]
    hours_sleep = sum(sleep_stage<5)/fs/3600
    
    data, hypoxia_burden = compute_hypoxia_burden(data, fs, hours_sleep=hours_sleep, apnea_name = 'apnea')
    
    T90burden, T90desaturation, T90nonspecific = hypoxaemic_burden_minutes(data['spo2'].values, fs)
    
    AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)

    return hypoxia_burden, AHI, hours_sleep, T90burden, T90desaturation, T90nonspecific


In [7]:
sleeplab_table.shape

(8000, 12)

In [12]:
sleeplab_table['Total_Sleep_Time'] = np.nan
sleeplab_table['AHI'] = np.nan
sleeplab_table['HypoxiaBurden'] = np.nan
sleeplab_table['T90_minutes'] = np.nan
sleeplab_table['T90_minutes_desat'] = np.nan
sleeplab_table['T90_minutes_unspecific'] = np.nan

for jloc, row in sleeplab_table.iterrows():
    
    try:
        if pd.isna(row.path_signal) | pd.isna(row.path_annotations):
            continue
        # read in raw data:
        signal, params = load_mgh_signal(row.path_signal)
        annotations = pd.read_csv(row.path_annotations)

        fs = int(params['Fs'])
        signal_len = signal.shape[0]

        # get respiratory event and sleep stage array:
        annotations = annotations_preprocess(annotations, fs)
        resp = vectorize_respiratory_events(annotations, signal_len)
        sleep_stage = vectorize_sleep_stages(annotations, signal_len)

        hypoxia_burden, ahi, total_sleep_time, T90burden, T90desaturation, T90nonspecific = compute_hypoxia_statistics(signal, resp, sleep_stage, fs)

        sleeplab_table.loc[jloc, 'AHI'] = ahi
        sleeplab_table.loc[jloc, 'HypoxiaBurden'] = hypoxia_burden
        sleeplab_table.loc[jloc, 'Total_Sleep_Time'] = total_sleep_time
        sleeplab_table.loc[jloc, 'T90_minutes'] = T90burden
        sleeplab_table.loc[jloc, 'T90_minutes_desat'] = T90desaturation
        sleeplab_table.loc[jloc, 'T90_minutes_unspecific'] = T90nonspecific

        sleeplab_table.to_csv(f'sleeplab_table_hypoxia_{part}.csv')
        
    except Exception as e:
        print(f'{jloc}, {row.Path}')
        print(e)

sleeplab_table.to_csv(f'sleeplab_table_hypoxia_{part}.csv')

/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a

81, M:\Datasets_ConvertedData\sleeplab\grass_data\Hexley_Allie_121015_0834.000
local variable 'samples_limit' referenced before assignment


/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a

134, M:\Datasets_ConvertedData\sleeplab\grass_data\Rummel_Jeremy_010811_0553.000
Cannot convert non-finite values (NA or inf) to integer


/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,


...

/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a

7979, M:\Datasets_ConvertedData\sleeplab\grass_data\Pradko_Meghan_052814_2046.000
index 5312800 is out of bounds for axis 0 with size 5312800


/home/wolfgang/repos/ICU-Sleep/code1/sleep_analysis_functions.py:436: RuntimeWarning: Mean of empty slice
  mean_spo2 = np.nanmean(np.array(spo2_collection), axis=0)


7989, M:\Datasets_ConvertedData\sleeplab\grass_data\O'Boy_Francis_040710_2123.000
Cannot convert non-finite values (NA or inf) to integer


/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,


In [9]:
sleeplab_table

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific
0,Anatasia_Lori_091611_2209.000,Z11530888,424-66-43,Anastasia,Lori E,Female,1972-08-11,2011-09-16,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.641667,0.3,0.7,0.000000,0.000000,0.000000
1,Gronewold_Catherine W_041615_2224.000,Z8592929,237-74-50,Gronewold,Catherine,Female,1948-12-13,2015-04-16,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.816667,26.1,33.7,16.903167,13.744833,3.158333
2,Morris_Martha_112512_2118.000,Z9992290,512-71-58,Morris,Martha,Female,1947-04-30,2012-11-25,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,0.883333,0.0,0.0,5.814500,4.109917,1.704583
3,Schlein_Toby_012717_2300.000,Z6488130,354-22-12,Schlein,Toby,Female,1945-11-16,2017-01-27,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,5.350000,8.4,6.4,6.542250,6.542250,0.000000
4,Callahan_Marian_042010_2158.000,Z10588347,160-39-24,Callahan,Marian,Female,1959-08-30,2010-04-20,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,Simmons_Winifred_110413_2255.000,Z9745908,531-71-07,Simmons,Winifred,Female,1959-02-28,2013-11-04,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7996,Dinezio_Charles J_010912_2300.000,Z8063076,220-71-68,Dinezio,Charles,Male,1930-07-25,2012-01-09,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN
7997,Thibadeaux_April_031013_2046.000,Z10767410,516-37-35,Thibadeaux,April R,Female,1980-04-12,2013-03-10,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7998,Eliott_Dean_082516_0912.000,Z6818312,506-90-16,Eliott,Dean,Male,1962-05-11,2016-08-25,MSLT,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

# ax[0].plot(resp)
# ax[0].plot(sleep_stage)
# ax[1].plot(signal['spo2'])
# ax[1].set_ylim([80,100])
# # ax[2].plot(signal['ptaf'])
# ax[3].plot(signal['chest'])
# ax[3].plot(signal['abd'])